# atari_200m_deterministic: Dopamine suite

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from config import EXPERIMENT

from analysis.curves import lifetime_average, return_curve
from analysis.stats import ATARI_REFERENCE_SCORES, human_normalize, mean_ci
from main import load_results

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots" / "dopamine_suite"

In [ ]:
# Load the DQN runs of every game with both a Dopamine curve and human and random
# scores, and Dopamine's published DQN curves.
component = EXPERIMENT.component("dqn_atari")
reference = json.loads(Path("dopamine_dqn_reference.json").read_text())
EXCLUDED = {"video_pinball"}  # temporarily removed as it dominates the graph
GAMES = sorted(
    game
    for game in component.sweep["ENV_HYPERS.GAME"]
    if game in reference and game in ATARI_REFERENCE_SCORES and game not in EXCLUDED
)

results = {
    game: load_results(EXPERIMENT, "dqn_atari", where={"ENV_HYPERS.GAME": game})
    for game in GAMES
}

In [ ]:
# Human normalize each game's lifetime average return. Dopamine's is the mean of
# its per-iteration returns, which are already averaged over its 5 seeds.
ours = np.array([
    human_normalize(
        lifetime_average(return_curve(results[game].reward, results[game].done)).mean(),
        game,
    )
    for game in GAMES
])
dopamine = np.array([
    human_normalize(np.mean(reference[game]["mean_return"]), game) for game in GAMES
])
improvement = ours - dopamine

In [ ]:
# Style for both bar plots.
BAR_STYLE = {
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
}
SUMMARY_SIZE = (6, 6)
IMPROVEMENT_SIZE = (20, 7)
SUMMARY_BARS = {"DQN": ("tab:blue", ours), "Dopamine Ref": ("black", dopamine)}
CAPSIZE = 8
HNS_LABEL = "Human normalized AUC"
IMPROVEMENT_LABEL = "Improvement over Dopamine Ref"
COLORMAP = "berlin"


def title(game):
    return game.replace("_", " ").title()

In [ ]:
# Plot the mean human normalized AUC across games, with a 95% bootstrap CI.
with plt.rc_context(BAR_STYLE):
    summary, ax = plt.subplots(figsize=SUMMARY_SIZE)
    for x, (color, scores) in enumerate(SUMMARY_BARS.values()):
        mean, low, high = mean_ci(scores)
        ax.bar(x, mean, color=color)
        ax.errorbar(x, mean, yerr=[[mean - low], [high - mean]], color="grey",
                    capsize=CAPSIZE)
    ax.set_xticks(range(len(SUMMARY_BARS)), SUMMARY_BARS.keys())
    ax.set(title=f"{len(GAMES)} games", ylabel=HNS_LABEL)
    summary.tight_layout()
plt.show()

In [ ]:
# Plot each game's improvement over Dopamine, sorted and colored by rank.
order = np.argsort(improvement)
colors = plt.get_cmap(COLORMAP)(np.linspace(0, 1, len(GAMES)))
with plt.rc_context(BAR_STYLE):
    per_game, ax = plt.subplots(figsize=IMPROVEMENT_SIZE)
    ax.bar(range(len(GAMES)), improvement[order], color=colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(range(len(GAMES)), [title(GAMES[i]) for i in order], rotation=90)
    ax.set_xlim(-0.5, len(GAMES) - 0.5)
    ax.set_ylabel(IMPROVEMENT_LABEL)
    per_game.tight_layout()
plt.show()

In [ ]:
# Save both plots as PDF and PNG in the plots directory.
figures = {"dopamine_suite_hns": summary, "dopamine_suite_improvement": per_game}
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")